In [1]:
import cProfile, pstats, time
from pathlib import Path
import pandas as pd
from mma3001_project.load import parse_env_csv
from mma3001_project.reconstruction import evaluate_methods, summarise_errors

ROOT = Path("..")
RAW = ROOT / "data/raw/5EnvSensor_MayToDec2024_180kRows.csv"

def best_time(f, repeats=3):
    """Run f() several times; return the fastest wall-clock time [s] and the output."""
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        out = f()
        times.append(time.perf_counter() - t0)
    return min(times), out

In [2]:
raw = pd.read_csv(RAW, dtype=str)
n_rows = len(raw)
n_distinct = len(raw.drop_duplicates(["sensorid", "jsondata"]))
print(f"{n_rows} uploaded rows, {n_distinct} distinct (sensor, payload) pairs "
      f"({100 * (1 - n_distinct / n_rows):.0f}% are repeats)")

t_base, long_base = best_time(lambda: parse_env_csv(RAW, dedupe_payloads=False))
t_fast, long_fast = best_time(lambda: parse_env_csv(RAW, dedupe_payloads=True))
pd.testing.assert_frame_equal(long_base, long_fast)
print(f"baseline {t_base:.2f} s, improved {t_fast:.2f} s, speed-up {t_base / t_fast:.1f}x; outputs identical")

179447 uploaded rows, 42847 distinct (sensor, payload) pairs (76% are repeats)
baseline 10.98 s, improved 3.79 s, speed-up 2.9x; outputs identical


In [3]:
prof = cProfile.Profile()
prof.enable()
parse_env_csv(RAW, dedupe_payloads=False)
prof.disable()
pstats.Stats(prof).sort_stats("tottime").print_stats(6)

         6521854 function calls (6521683 primitive calls) in 8.624 seconds

   Ordered by: internal time
   List reduced from 735 to 6 due to restriction <6>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
   179447    2.337    0.000    2.337    0.000 C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0\Lib\json\decoder.py:351(raw_decode)
        1    1.900    1.900    1.988    1.988 c:\Users\helen\OneDrive\Desktop\MMA3001proj\MMA3001-project\.venv\Lib\site-packages\pandas\io\parsers\c_parser_wrapper.py:226(read)
        2    1.167    0.584    5.741    2.871 C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0\Lib\asyncio\base_events.py:1980(_run_once)
        1    0.465    0.465    2.730    2.730 C:\Users\helen\OneDrive\Desktop\MMA3001proj\MMA3001-project\src\mma3001_project\load.py:37(parse_env_csv)
        5    0.395    0.079    0.451    0.090 c:\Users\helen\OneDrive\Desk

In [4]:
clean = pd.read_csv(ROOT / "data/processed/sensor869_clean.csv", index_col="time_utc", parse_dates=True)
dev = clean["co2_ppm"][clean.index < pd.Timestamp("2024-06-01", tz="Australia/Melbourne")]

t_whole, err_whole = best_time(lambda: evaluate_methods(dev, [1, 2], n_gaps=200, seed=1, window=len(dev)), repeats=1)
t_win, err_win = best_time(lambda: evaluate_methods(dev, [1, 2], n_gaps=200, seed=1, window=72))
print(f"whole series {t_whole:.2f} s, 12-h window {t_win:.2f} s, speed-up {t_whole / t_win:.1f}x")

both = pd.concat({"whole series": summarise_errors(err_whole)["RMSE"],
                  "12-h window": summarise_errors(err_win)["RMSE"]}, axis=1)
both["difference"] = both["12-h window"] - both["whole series"]
both.round(3)

whole series 1.48 s, 12-h window 0.50 s, speed-up 3.0x


whole series  12-h window  difference
method length                                       
cubic  1             12.505       12.505        -0.0
       2             14.057       14.057         0.0
linear 1             10.360       10.360        -0.0
       2             10.799       10.799         0.0
locf   1             12.332       12.332         0.0
       2             13.332       13.332         0.0
pchip  1             10.640       10.640        -0.0
       2             11.178       11.178         0.0

In [5]:
perf = pd.DataFrame({"baseline [s]": [t_base, t_whole], "improved [s]": [t_fast, t_win]},
                    index=["parse raw file (dedupe payloads)", "validate methods (12-h window)"])
perf["speed-up"] = perf["baseline [s]"] / perf["improved [s]"]
perf.to_csv(ROOT / "results/performance.csv")
perf.round(2)

,baseline [s],improved [s],speed-up
parse raw file (dedupe payloads),10.98,3.79,2.90
validate methods (12-h window),1.48,0.50,2.99
